In [0]:
# =============================================================================
# Notebook  : 02b_map_07_contacts_attributes
# Location  : /HGI-Lakehouse-Pipeline/03_Silver_Layer/02b_map_07_contacts_attributes
# Spec Ref  : §1.9 contacts_attributes (contact_id, is_paying, is_excluded, mrr)
# Purpose   : Create one attributes row for EVERY contact.
# Run after : map_01 (contacts_all)
# =============================================================================

In [0]:
# CELL 1 ── Load shared config
# In Databricks: each %run must be in its own cell
%run ../config/pipeline_config.py

In [0]:
%run ./_map_common.py

In [0]:
# CELL 2
dbutils.widgets.text("customer_id", "cust_001")
customer_id = dbutils.widgets.get("customer_id").strip().lower()
print(f"=== Map 07: contacts_attributes ===  customer={customer_id}")

In [0]:
# CELL 3
create_map_table(
    f"{sv}.contacts_attributes",
    """
        contact_id   STRING NOT NULL,
        tenant       BIGINT,
        is_paying    BOOLEAN,
        is_excluded  BOOLEAN,
        mrr          DOUBLE
    """,
    cluster_by="contact_id"
)

In [0]:
# CELL 4

# Safe drop in case target exists as a VIEW
safe_drop_view(f"{sv}.contacts_attributes")

spark.sql(f"""
    CREATE OR REPLACE TABLE {sv}.contacts_attributes
    USING DELTA
    CLUSTER BY (contact_id)
    {DELTA_TBLPROPS_MAP}
    AS
    SELECT
        id      AS contact_id,
        tenant,
        false   AS is_paying,
        false   AS is_excluded,
        0.0     AS mrr
    FROM {sv}.contacts_all
""")

n = cnt(f"{sv}.contacts_attributes")
print(f"  contacts_attributes: {n:,} rows")
dbutils.notebook.exit("Success")
